### SS3 Part B — STADIOalot Order Return Prediction

#### CAP182 — SS3 Part B

This notebook applies the final return-prediction method selected during SS2 to the supplied STADIOalot client data. The final method selected in SS2 was an XGBoost binary classification model. The prediction unit for this stage is an individual order.

The notebook is structured according to the required components:

1. Dataset Loading
2. Dataset Preprocessing
3. Model Training
4. Model Validation
5. Results Visualisation
6. Final Results and Conclusion

### 1. Dataset Loading

The client extract consists of eight related CSV tables covering orders placed between January 2024 and June 2026.
The eight tables are:

1. `table1_orders_transactions.csv`
2. `table2_returns.csv`
3. `table3_product_catalogue.csv`
4. `table4_marketplace_sellers.csv`
5. `table5_customer_information.csv`
6. `table6_customer_behaviour.csv`
7. `table7_inventory_warehouse.csv`
8. `table8_delivery_logistics.csv`

The data is supplied as an uncleaned operational extract in comma-separated values (CSV) format. CSV is suitable for these tabular extracts as it is a widely supported, portable plain-text format that pandas can load directly for inspection, cleaning and analysis. The eight tables are loaded separately first as each represents a different operational subject; they are then checked and joined using the relevant common keys during preprocessing. The agreed prediction unit is an order. An order is labelled as:

- `1` — Returned: at least one item from the order was returned within the 30-day observation window.
- `0` — Not Returned: no return event is recorded for the order.

The return table is used to construct the target variable however it is not used as a predictive input as its information is only available after the return has occurred.

In [50]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE DISPLAY
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

from xgboost import XGBClassifier

import joblib

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 100)

# Make tables easier to read
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


# ============================================================
# 2. SET RANDOM SEED
# ============================================================

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


print("Libraries imported successfully.")

Libraries imported successfully.


In [51]:
# ============================================================
# 3. DEFINE CLIENT DATA PATH
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ClientData"

print("PROJECT ROOT:")
print(PROJECT_ROOT.resolve())

print("\nFULL CLIENT DATA DIRECTORY:")
print(DATA_PATH.resolve())

# ============================================================
# DEFINE CLIENT DATA FILES
# ============================================================

client_filenames = [
    "table1_orders_transactions.csv",
    "table2_returns.csv",
    "table3_product_catalogue.csv",
    "table4_marketplace_sellers.csv",
    "table5_customer_information.csv",
    "table6_customer_behaviour.csv",
    "table7_inventory_warehouse.csv",
    "table8_delivery_logistics.csv"
]
# Display filenames and their full directory paths in a table

file_check = pd.DataFrame({
    "Dataset": client_filenames,
    "Full Directory": [
        str((DATA_PATH / filename).resolve())
        for filename in client_filenames
    ],
    "Status": [
        "FOUND" if (DATA_PATH / filename).is_file()
        else "NOT FOUND"
        for filename in client_filenames
    ]
})

print("\nFULL CLIENT DATA DIRECTORY:")
print(DATA_PATH.resolve())


# ============================================================
# CLIENT FILE CHECK — DISPLAY FULL DIRECTORY PATHS
# ============================================================

# Prevent pandas from truncating long directory paths
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Build the file-check table
file_check = pd.DataFrame({
    "Dataset": client_filenames,
    "Full Directory": [
        str((DATA_PATH / filename).resolve())
        for filename in client_filenames
    ],
    "Status": [
        "FOUND" if (DATA_PATH / filename).is_file()
        else "NOT FOUND"
        for filename in client_filenames
    ]
})

print("\nCLIENT FILE CHECK:")

# Display the complete paths inside the table
display(file_check)

# ============================================================
# VALIDATE CLIENT DATA DIRECTORY AND FILES
# ============================================================

if not DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"Client data directory not found: {DATA_PATH.resolve()}"
    )

missing_files = [
    filename for filename in client_filenames
    if not (DATA_PATH / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following files could not be found:\n"
        + "\n".join(missing_files)
    )

print("\nAll eight client files have been found.")

# ============================================================
# 4. LOAD ALL EIGHT CLIENT TABLES
# ============================================================

table1_orders = pd.read_csv(DATA_PATH / client_filenames[0])
table2_returns = pd.read_csv(DATA_PATH / client_filenames[1])
table3_products = pd.read_csv(DATA_PATH / client_filenames[2])
table4_sellers = pd.read_csv(DATA_PATH / client_filenames[3])
table5_customers = pd.read_csv(DATA_PATH / client_filenames[4])
table6_behaviour = pd.read_csv(DATA_PATH / client_filenames[5])
table7_inventory = pd.read_csv(DATA_PATH / client_filenames[6])
table8_delivery = pd.read_csv(DATA_PATH / client_filenames[7])

print("\nAll eight client tables loaded successfully.")


# ============================================================
# 5. DISPLAY TABLE STRUCTURE
# ============================================================

tables = {
    "Table 1 - Orders and Transactions": table1_orders,
    "Table 2 - Returns": table2_returns,
    "Table 3 - Product Catalogue": table3_products,
    "Table 4 - Marketplace Sellers": table4_sellers,
    "Table 5 - Customer Information": table5_customers,
    "Table 6 - Customer Behaviour": table6_behaviour,
    "Table 7 - Inventory and Warehouse": table7_inventory,
    "Table 8 - Delivery and Logistics": table8_delivery
}

# Summarise the structure of each client dataset
table_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns)
    }
    for name, df in tables.items()
])

print("\nCLIENT DATASET STRUCTURE:")
display(table_summary)

# ============================================================
# 6. DISPLAY COLUMN NAMES IN TABLES
# ============================================================

for name, df in tables.items():

    print(f"\n{name}")

    column_table = pd.DataFrame({
        "Column Number": range(1, len(df.columns) + 1),
        "Column Name": df.columns
    })

    display(column_table)

PROJECT ROOT:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction

FULL CLIENT DATA DIRECTORY:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData

FULL CLIENT DATA DIRECTORY:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData

CLIENT FILE CHECK:


,Dataset,Full Directory,Status
0,table1_orders_transactions.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table1_orders_transactions.csv,FOUND
1,table2_returns.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table2_returns.csv,FOUND
2,table3_product_catalogue.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table3_product_catalogue.csv,FOUND
3,table4_marketplace_sellers.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table4_marketplace_sellers.csv,FOUND
4,table5_customer_information.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table5_customer_information.csv,FOUND
5,table6_customer_behaviour.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table6_customer_behaviour.csv,FOUND
6,table7_inventory_warehouse.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table7_inventory_warehouse.csv,FOUND
7,table8_delivery_logistics.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\raw\ClientData\table8_delivery_logistics.csv,FOUND



All eight client files have been found.

All eight client tables loaded successfully.

CLIENT DATASET STRUCTURE:


,Dataset,Rows,Columns
0,Table 1 - Orders and Transactions,11841,11
1,Table 2 - Returns,2037,11
2,Table 3 - Product Catalogue,11818,11
3,Table 4 - Marketplace Sellers,8241,7
4,Table 5 - Customer Information,8241,11
5,Table 6 - Customer Behaviour,8241,9
6,Table 7 - Inventory and Warehouse,11818,7
7,Table 8 - Delivery and Logistics,8241,10



Table 1 - Orders and Transactions


...

,Column Number,Column Name
0,1,order_id
1,2,customer_id
2,3,look_back_period_days
3,4,sessions
4,5,product_views
5,6,add_to_basket_count
6,7,previous_orders
7,8,previous_returns
8,9,customer_return_rate



Table 7 - Inventory and Warehouse


,Column Number,Column Name
0,1,order_id
1,2,sku
2,3,distribution_centre
3,4,stock_on_hand
4,5,stock_age_days
5,6,historical_pick_pack_time
6,7,inbound_datetime



Table 8 - Delivery and Logistics


,Column Number,Column Name
0,1,order_id
1,2,estimated_delivery_datetime
2,3,delivery_option
3,4,delivery_province
4,5,route_distance_km
5,6,historical_delivery_time_days
6,7,historical_delivery_failure_rate
7,8,actual_delivery_datetime
8,9,delivery_attempts
9,10,on_time_flag


In [52]:
# ============================================================
# 7. CREATE ORDER-LEVEL TARGET
# ============================================================

# Obtain the distinct list of orders from the main order table.
orders = table1_orders[["order_id"]].drop_duplicates().copy()

# Identify orders with at least one return
returned_orders = (
    table2_returns[["order_id"]]
    .drop_duplicates()
    .assign(Return_Status=1)
)

# Left join returns onto all orders
orders_with_target = orders.merge(
    returned_orders,
    on="order_id",
    how="left"
)

# Orders without a return record are classified as 0.
orders_with_target["Return_Status"] = (
    orders_with_target["Return_Status"]
    .fillna(0)
    .astype(int)
)

print("Order-level target created successfully.")
print(f"Total unique orders: {len(orders_with_target):,}")

Order-level target created successfully.
Total unique orders: 8,241


In [53]:
# ============================================================
# 8. CLASS DISTRIBUTION AND CLASS IMBALANCE
# ============================================================

# Calculate class counts
class_counts = (
    orders_with_target["Return_Status"]
    .value_counts()
    .sort_index()
)

# Calculate class percentages
class_percentages = (
    orders_with_target["Return_Status"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

# Extract class counts
not_returned = class_counts.get(0, 0)
returned = class_counts.get(1, 0)

# Calculate imbalance ratio
if returned > 0:
    imbalance_ratio = not_returned / returned
else:
    imbalance_ratio = np.nan

# Create combined summary table
class_summary = pd.DataFrame({
    "Class": [
        "0 - Not Returned",
        "1 - Returned",
        "Class Imbalance"
    ],
    "Count": [
        not_returned,
        returned,
        np.nan
    ],
    "Percentage": [
        class_percentages.get(0, 0),
        class_percentages.get(1, 0),
        np.nan
    ],
    "Imbalance Ratio": [
        np.nan,
        np.nan,
        imbalance_ratio
    ]
})

# Display combined table
print("\nCLASS DISTRIBUTION AND IMBALANCE SUMMARY:")

display(
    class_summary.style
    .format({
        "Count": lambda x: f"{x:,.0f}" if pd.notna(x) else "—",
        "Percentage": lambda x: f"{x:.2f}%" if pd.notna(x) else "—",
        "Imbalance Ratio": lambda x: f"{x:.2f}:1" if pd.notna(x) else "—"
    })
    .hide(axis="index")
    .set_caption("Order-Level Target Distribution and Class Imbalance")
)


CLASS DISTRIBUTION AND IMBALANCE SUMMARY:


Class,Count,Percentage,Imbalance Ratio
0 - Not Returned,"6,372",77.32%,—
1 - Returned,"1,869",22.68%,—
Class Imbalance,—,—,3.41:1


### Dataset Loading Summary

The eight client tables have been loaded separately as they contain different types of information linked through keys such as `order_id`, `customer_id`, `seller_id` and `sku`. The modelling dataset contains **8,241 unique orders**, with one row per order, as the business question is to predict whether an order will result in a return.

The target variable is `Return_Status`:

- **Class 0 — Not Returned:** 6,372 orders (**77.32%**).
- **Class 1 — Returned:** 1,869 orders (**22.68%**).

The class imbalance ratio is approximately **3.41:1** (not-returned to returned orders). Returned orders are therefore the minority class. This matters when interpreting performance as a model can achieve seemingly high accuracy by mostly predicting the majority class; precision, recall, F1-score and ROC-AUC are also reported.

**Format justification:** CSV is appropriate for the supplied structured tables as it preserves rows and columns in a portable format and can be read directly by `pandas.read_csv()`. Loading each file separately also allows columns, data types, missing values and join keys to be checked before the tables are combined.

**Target construction:** `table2_returns.csv` is used to identify orders with a return record and construct `Return_Status`; it is not used as a predictive input as it contains information about the outcome itself.

### 2. Dataset Preprocessing

The supplied client extract is not pre-cleaned. Therefore, preprocessing is required before model training.
The preprocessing stage performs the following tasks:

1. Standardise column data types where required.
2. Convert date and datetime fields into usable representations.
3. Construct the order-level target.
4. Aggregate item-level tables to one row per order.
5. Combine the eight data sources using their agreed keys.
6. Remove identifiers that should not be used as predictive features.
7. Remove information that becomes available only after the prediction point.
8. Create useful order-level temporal and numerical features.
9. Handle missing numerical and categorical values.
10. Encode categorical variables.
11. Split the data into training, validation and unseen test subsets.

The prediction point is at the order stage. Therefore, only information available at or before the order timestamp is retained as a predictor. `table2_returns.csv` is used to construct the target but is not used as a predictor. In `table8_delivery_logistics.csv`, `actual_delivery_datetime`, `delivery_attempts` and `on_time_flag` are excluded as they are post-purchase information.

In [54]:
# ============================================================
# 9. STANDARDISE IMPORTANT DATA TYPES
# ============================================================

# ------------------------------------------------------------
# Orders
# ------------------------------------------------------------
table1_orders["order_datetime"] = pd.to_datetime(
    table1_orders["order_datetime"],
    errors="coerce"
)

table1_orders["quantity"] = pd.to_numeric(
    table1_orders["quantity"],
    errors="coerce"
)

table1_orders["unit_price"] = pd.to_numeric(
    table1_orders["unit_price"],
    errors="coerce"
)

table1_orders["promotion_amount"] = pd.to_numeric(
    table1_orders["promotion_amount"],
    errors="coerce"
)

# ------------------------------------------------------------
# Product catalogue
# ------------------------------------------------------------
for column in [
    "product_price",
    "product_rating",
    "review_count",
    "listing_age_days"
]:
    table3_products[column] = pd.to_numeric(
        table3_products[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# Marketplace sellers
# ------------------------------------------------------------
for column in [
    "seller_rating",
    "seller_tenure_months",
    "average_shipping_days",
    "late_shipment_rate",
    "complaint_count"
]:
    table4_sellers[column] = pd.to_numeric(
        table4_sellers[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# Customer information
# ------------------------------------------------------------
for column in [
    "customer_age",
    "customer_tenure_months",
    "average_order_value",
    "total_historical_spend",
    "previous_returned_orders"
]:
    table5_customers[column] = pd.to_numeric(
        table5_customers[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# Customer behaviour
# ------------------------------------------------------------
for column in [
    "look_back_period_days",
    "sessions",
    "product_views",
    "add_to_basket_count",
    "previous_orders",
    "previous_returns",
    "customer_return_rate"
]:
    table6_behaviour[column] = pd.to_numeric(
        table6_behaviour[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# Inventory
# ------------------------------------------------------------
for column in [
    "stock_on_hand",
    "stock_age_days",
    "historical_pick_pack_time"
]:
    table7_inventory[column] = pd.to_numeric(
        table7_inventory[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# Delivery
# ------------------------------------------------------------
for column in [
    "route_distance_km",
    "historical_delivery_time_days",
    "historical_delivery_failure_rate"
]:
    table8_delivery[column] = pd.to_numeric(
        table8_delivery[column],
        errors="coerce"
    )

print("Important data types standardised.")

Important data types standardised.


In [55]:
# ============================================================
# 10. DATA QUALITY CHECKS
# ============================================================

def data_quality_summary(df, table_name):
    summary = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.astype(str).values,
        "Missing Values": df.isna().sum().values,
        "Missing %": (
            df.isna().mean().values * 100
        ),
        "Unique Values": [
            df[col].nunique(dropna=True)
            for col in df.columns
        ]
    })

    display(
        summary.style
        .format({"Missing %": "{:.2f}%"})
        .set_caption(f"Data Quality — {table_name}")
    )

for name, df in tables.items():
    data_quality_summary(df, name)

,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,customer_id,str,0,0.00%,4092
2,order_datetime,datetime64[us],126,1.06%,8179
3,sku,str,0,0.00%,1204
4,seller_id,str,0,0.00%,147
5,quantity,int64,0,0.00%,3
6,unit_price,float64,0,0.00%,1215
7,promotion_amount,float64,0,0.00%,3003
8,payment_method,str,0,0.00%,25
9,delivery_option,str,0,0.00%,20


,Column,Data Type,Missing Values,Missing %,Unique Values
0,return_id,str,0,0.00%,2037
1,order_id,str,0,0.00%,1869
2,return_datetime,str,0,0.00%,2037
3,return_receipt_datetime,str,0,0.00%,2037
4,returned_quantity,int64,0,0.00%,3
5,return_reason,str,0,0.00%,20
6,return_condition,str,0,0.00%,14
7,refund_amount,float64,0,0.00%,2030
8,refund_outcome,str,0,0.00%,5
9,return_cost,float64,0,0.00%,1882


,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,sku,str,0,0.00%,1204
2,product_category,str,0,0.00%,41
3,subcategory,str,0,0.00%,142
4,brand,str,0,0.00%,20
5,size,str,7049,59.65%,16
6,colour,str,0,0.00%,55
7,product_price,float64,0,0.00%,1194
8,product_rating,float64,480,4.06%,35
9,review_count,int64,0,0.00%,1100


,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,seller_id,str,0,0.00%,147
2,seller_rating,float64,0,0.00%,22
3,seller_tenure_months,int64,0,0.00%,97
4,average_shipping_days,float64,0,0.00%,39
5,late_shipment_rate,float64,0,0.00%,107
6,complaint_count,int64,0,0.00%,44


,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,customer_id,str,0,0.00%,4092
2,customer_age,float64,47,0.57%,61
3,customer_gender,str,0,0.00%,15
4,customer_province,str,0,0.00%,24
5,account_creation_date,str,0,0.00%,4092
6,customer_tenure_months,int64,0,0.00%,78
7,preferred_shopping_channel,str,0,0.00%,15
8,average_order_value,float64,4092,49.65%,3769
9,total_historical_spend,float64,0,0.00%,3776


,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,customer_id,str,0,0.00%,4092
2,look_back_period_days,int64,0,0.00%,1
3,sessions,int64,0,0.00%,27
4,product_views,int64,0,0.00%,152
5,add_to_basket_count,int64,0,0.00%,21
6,previous_orders,int64,0,0.00%,7
7,previous_returns,int64,0,0.00%,6
8,customer_return_rate,float64,0,0.00%,12


,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,sku,str,0,0.00%,1204
2,distribution_centre,str,0,0.00%,5
3,stock_on_hand,int64,0,0.00%,584
4,stock_age_days,int64,0,0.00%,196
5,historical_pick_pack_time,float64,0,0.00%,166
6,inbound_datetime,str,0,0.00%,11818


,Column,Data Type,Missing Values,Missing %,Unique Values
0,order_id,str,0,0.00%,8241
1,estimated_delivery_datetime,str,0,0.00%,8239
2,delivery_option,str,0,0.00%,5
3,delivery_province,str,0,0.00%,9
4,route_distance_km,float64,0,0.00%,675
5,historical_delivery_time_days,float64,0,0.00%,57
6,historical_delivery_failure_rate,float64,0,0.00%,85
7,actual_delivery_datetime,str,0,0.00%,8241
8,delivery_attempts,int64,0,0.00%,2
9,on_time_flag,int64,0,0.00%,2


In [56]:
# ============================================================
# 11. AGGREGATE TABLE 1 — ORDERS AND TRANSACTIONS
# ============================================================

# Create line-level monetary value first.
table1_orders["line_gross_value"] = (
    table1_orders["quantity"] *
    table1_orders["unit_price"]
)

table1_orders["net_line_value"] = (
    table1_orders["line_gross_value"]
    - table1_orders["promotion_amount"].fillna(0)
)


# Aggregate item-level information to one row per order.
orders_features = (
    table1_orders
    .groupby("order_id", as_index=False)
    .agg(
        customer_id=("customer_id", "first"),
        order_datetime=("order_datetime", "first"),

        item_count=("sku", "nunique"),
        total_quantity=("quantity", "sum"),

        order_gross_value=("line_gross_value", "sum"),
        total_promotion_amount=("promotion_amount", "sum"),
        order_net_value=("net_line_value", "sum"),

        average_unit_price=("unit_price", "mean"),

        payment_method=("payment_method", "first"),
        delivery_option=("delivery_option", "first"),
        customer_province=("customer_province", "first")
    )
)


print("Table 1 aggregated to order level.")
print(f"Orders represented: {len(orders_features):,}")

print("\nDisplay Orders Features")
display(orders_features.head())

# ============================================================
# 12. AGGREGATE TABLE 3 — PRODUCT CATALOGUE
# ============================================================

products_features = (
    table3_products
    .groupby("order_id", as_index=False)
    .agg(
        average_product_price=("product_price", "mean"),
        average_product_rating=("product_rating", "mean"),
        average_review_count=("review_count", "mean"),
        average_listing_age_days=("listing_age_days", "mean"),

        product_category=("product_category", "first"),
        product_subcategory=("subcategory", "first"),
        product_brand=("brand", "first"),
        product_size=("size", "first"),
        product_colour=("colour", "first")
    )
)


print("Table 3 aggregated to order level.")
print(f"Orders represented: {len(products_features):,}")

print("\nDisplay Products Features")
display(products_features.head())

# ============================================================
# 13. AGGREGATE TABLE 7 — INVENTORY AND WAREHOUSE
# ============================================================

inventory_features = (
    table7_inventory
    .groupby("order_id", as_index=False)
    .agg(
        average_stock_on_hand=("stock_on_hand", "mean"),
        minimum_stock_on_hand=("stock_on_hand", "min"),

        average_stock_age_days=("stock_age_days", "mean"),

        average_pick_pack_time=("historical_pick_pack_time", "mean"),

        distribution_centre=("distribution_centre", "first")
    )
)


print("Table 7 aggregated to order level.")
print(f"Orders represented: {len(inventory_features):,}")

print("\nDisplay Inventory Features")
display(inventory_features.head())

# ============================================================
# 14. AGGREGATE TABLE 4 — MARKETPLACE SELLERS
# ============================================================

seller_features = (
    table4_sellers
    .groupby("order_id", as_index=False)
    .agg(
        average_seller_rating=("seller_rating", "mean"),
        average_seller_tenure_months=("seller_tenure_months", "mean"),
        average_shipping_days=("average_shipping_days", "mean"),
        average_late_shipment_rate=("late_shipment_rate", "mean"),
        total_seller_complaints=("complaint_count", "sum")
    )
)

print("Display Seller Features")
display(seller_features.head())

Table 1 aggregated to order level.
Orders represented: 8,241

Display Orders Features


,order_id,customer_id,order_datetime,item_count,total_quantity,order_gross_value,total_promotion_amount,order_net_value,average_unit_price,payment_method,delivery_option,customer_province
0,ORD840000,CUST20997,2026-01-06 09:07:48,1,3,816.42,0.00,816.42,272.14,Instant EFT,Standard,WC
1,ORD840001,CUST20407,2026-05-06 02:04:40,1,1,56.47,0.00,56.47,56.47,Instant EFT,Express,GP
2,ORD840002,CUST20793,2026-04-23 22:55:50,2,2,623.78,83.61,540.17,311.89,Wallet,Express,WC
3,ORD840003,CUST21163,2024-01-16 16:01:57,4,4,1545.93,344.97,1200.96,386.48,Card,Standard,GP
4,ORD840004,CUST20008,2024-06-14 07:53:53,1,1,864.55,0.00,864.55,864.55,Card,Express,FS


Table 3 aggregated to order level.
Orders represented: 8,241

Display Products Features


,order_id,average_product_price,average_product_rating,average_review_count,average_listing_age_days,product_category,product_subcategory,product_brand,product_size,product_colour
0,ORD840000,272.14,3.60,9.00,751.00,Baby & Kids,Nursery,Brand Q,S,Green
1,ORD840001,56.47,3.00,117.00,414.00,Baby & Kids,Feeding,Brand R,XXL,Olive
2,ORD840002,311.89,3.55,855.50,1082.50,Toys & Games,Outdoor Toys,Brand G,XS,Cream
3,ORD840003,386.48,3.83,0.00,0.00,Apparel,Women's Dresses,Brand C,XXL,Navy
4,ORD840004,864.55,3.60,256.00,823.00,Home & Kitchen,Lighting,Brand Q,NaN,Blue


Table 7 aggregated to order level.
Orders represented: 8,241

Display Inventory Features


,order_id,average_stock_on_hand,minimum_stock_on_hand,average_stock_age_days,average_pick_pack_time,distribution_centre
0,ORD840000,437.00,437,20.00,5.60,DC1
1,ORD840001,192.00,192,23.00,10.00,DC3
2,ORD840002,162.00,64,49.00,8.40,DC5
3,ORD840003,179.75,19,43.50,6.50,DC4
4,ORD840004,305.00,305,29.00,11.30,DC4


Display Seller Features


,order_id,average_seller_rating,average_seller_tenure_months,average_shipping_days,average_late_shipment_rate,total_seller_complaints
0,ORD840000,3.80,19.00,3.50,0.13,18
1,ORD840001,4.10,35.00,3.70,0.12,10
2,ORD840002,3.90,60.00,3.40,0.11,15
3,ORD840003,4.20,29.00,3.50,0.12,7
4,ORD840004,3.70,27.00,4.00,0.14,7


In [57]:
# ============================================================
# 15. TABLE 5 — CUSTOMER INFORMATION
# ============================================================

customer_features = (
    table5_customers
    .drop_duplicates(subset=["order_id"])
    .copy()
)


customer_features = customer_features[
    [
        "order_id",
        "customer_age",
        "customer_gender",
        "customer_province",
        "account_creation_date",
        "customer_tenure_months",
        "preferred_shopping_channel",
        "average_order_value",
        "total_historical_spend",
        "previous_returned_orders"
    ]
]


customer_features["account_creation_date"] = pd.to_datetime(
    customer_features["account_creation_date"],
    errors="coerce"
)

print("Display Customer Features")
display(customer_features.head())

# ============================================================
# 16. TABLE 6 — CUSTOMER BEHAVIOUR
# ============================================================

behaviour_features = (
    table6_behaviour
    .drop_duplicates(subset=["order_id"])
    .copy()
)


behaviour_features = behaviour_features[
    [
        "order_id",
        "look_back_period_days",
        "sessions",
        "product_views",
        "add_to_basket_count",
        "previous_orders",
        "previous_returns",
        "customer_return_rate"
    ]
]

print("Display Customer Behaviour")
display(behaviour_features.head())

# ============================================================
# 17. TABLE 8 — DELIVERY AND LOGISTICS
# ============================================================

delivery_features = (
    table8_delivery
    .drop_duplicates(subset=["order_id"])
    .copy()
)

# Convert dates.
delivery_features["estimated_delivery_datetime"] = pd.to_datetime(
    delivery_features["estimated_delivery_datetime"],
    errors="coerce"
)

delivery_features["actual_delivery_datetime"] = pd.to_datetime(
    delivery_features["actual_delivery_datetime"],
    errors="coerce"
)


# Keep only pre-purchase delivery information.
delivery_features = delivery_features[
    [
        "order_id",
        "estimated_delivery_datetime",
        "delivery_option",
        "delivery_province",
        "route_distance_km",
        "historical_delivery_time_days",
        "historical_delivery_failure_rate"
    ]
].copy()

print("Display Delivery Features")
display(delivery_features.head())

Display Customer Features


,order_id,customer_age,customer_gender,customer_province,account_creation_date,customer_tenure_months,preferred_shopping_channel,average_order_value,total_historical_spend,previous_returned_orders
0,ORD840000,35.00,Male,WC,2025-12-13 22:41:13,0,Mobile App,NaN,0.00,0
1,ORD840001,41.00,Female,GP,2020-10-06 02:11:44,66,Website,NaN,0.00,0
2,ORD840002,30.00,Other,WC,2020-12-03 09:54:40,64,Mobile App,NaN,0.00,0
3,ORD840003,67.00,Female,GP,2021-06-06 06:11:24,31,Website,NaN,0.00,0
4,ORD840004,39.00,Male,FS,2022-10-18 22:09:09,19,Website,NaN,0.00,0


Display Customer Behaviour


,order_id,look_back_period_days,sessions,product_views,add_to_basket_count,previous_orders,previous_returns,customer_return_rate
0,ORD840000,30,0,37,4,0,0,0.00
1,ORD840001,30,4,25,2,0,0,0.00
2,ORD840002,30,3,31,12,0,0,0.00
3,ORD840003,30,3,40,5,0,0,0.00
4,ORD840004,30,6,106,6,0,0,0.00


Display Delivery Features


,order_id,estimated_delivery_datetime,delivery_option,delivery_province,route_distance_km,historical_delivery_time_days,historical_delivery_failure_rate
0,ORD840000,2026-01-10 18:41:52,Standard,WC,33.40,2.20,0.11
1,ORD840001,2026-05-08 12:04:19,Express,GP,6.00,3.30,0.09
2,ORD840002,2026-04-26 13:40:05,Express,WC,25.70,1.40,0.07
3,ORD840003,2024-01-22 00:06:50,Standard,GP,5.00,2.80,0.12
4,ORD840004,2024-06-17 06:47:23,Express,FS,40.10,4.70,0.09


In [58]:
# ============================================================
# 18. MERGE ALL PREDICTIVE TABLES
# ============================================================

model_data = orders_features.copy()

# Product information
model_data = model_data.merge(
    products_features,
    on="order_id",
    how="left"
)

# Seller information
model_data = model_data.merge(
    seller_features,
    on="order_id",
    how="left"
)

# Customer information
model_data = model_data.merge(
    customer_features,
    on="order_id",
    how="left"
)

# Customer behaviour
model_data = model_data.merge(
    behaviour_features,
    on="order_id",
    how="left"
)

# Inventory information
model_data = model_data.merge(
    inventory_features,
    on="order_id",
    how="left"
)

# Delivery information
model_data = model_data.merge(
    delivery_features,
    on="order_id",
    how="left"
)

# Add target.
model_data = model_data.merge(
    orders_with_target,
    on="order_id",
    how="left"
)

print(f"Final modelling rows before feature preparation: {len(model_data):,}")
print(f"Final modelling columns: {len(model_data.columns):,}")

display(model_data.head())

# ============================================================
# 19. VERIFY ORDER-LEVEL MODELLING UNIT
# ============================================================

total_rows = len(model_data)
unique_orders = model_data["order_id"].nunique()

print(f"Total modelling rows : {total_rows:,}")
print(f"Unique order IDs     : {unique_orders:,}")

if total_rows == unique_orders:
    print("\nPASS: There is exactly one modelling row per order.")
else:
    print("\nWARNING: Duplicate order IDs remain.")

Final modelling rows before feature preparation: 8,241


Final modelling columns: 54


,order_id,customer_id,order_datetime,item_count,total_quantity,order_gross_value,total_promotion_amount,order_net_value,average_unit_price,payment_method,delivery_option_x,customer_province_x,average_product_price,average_product_rating,average_review_count,average_listing_age_days,product_category,product_subcategory,product_brand,product_size,product_colour,average_seller_rating,average_seller_tenure_months,average_shipping_days,average_late_shipment_rate,total_seller_complaints,customer_age,customer_gender,customer_province_y,account_creation_date,customer_tenure_months,preferred_shopping_channel,average_order_value,total_historical_spend,previous_returned_orders,look_back_period_days,sessions,product_views,add_to_basket_count,previous_orders,previous_returns,customer_return_rate,average_stock_on_hand,minimum_stock_on_hand,average_stock_age_days,average_pick_pack_time,distribution_centre,estimated_delivery_datetime,delivery_option_y,delivery_province,route_distance_km,historical_delivery_time_days,historical_delivery_failure_rate,Return_Status
0,ORD840000,CUST20997,2026-01-06 09:07:48,1,3,816.42,0.00,816.42,272.14,Instant EFT,Standard,WC,272.14,3.60,9.00,751.00,Baby & Kids,Nursery,Brand Q,S,Green,3.80,19.00,3.50,0.13,18,35.00,Male,WC,2025-12-13 22:41:13,0,Mobile App,NaN,0.00,0,30,0,37,4,0,0,0.00,437.00,437,20.00,5.60,DC1,2026-01-10 18:41:52,Standard,WC,33.40,2.20,0.11,0
1,ORD840001,CUST20407,2026-05-06 02:04:40,1,1,56.47,0.00,56.47,56.47,Instant EFT,Express,GP,56.47,3.00,117.00,414.00,Baby & Kids,Feeding,Brand R,XXL,Olive,4.10,35.00,3.70,0.12,10,41.00,Female,GP,2020-10-06 02:11:44,66,Website,NaN,0.00,0,30,4,25,2,0,0,0.00,192.00,192,23.00,10.00,DC3,2026-05-08 12:04:19,Express,GP,6.00,3.30,0.09,1
2,ORD840002,CUST20793,2026-04-23 22:55:50,2,2,623.78,83.61,540.17,311.89,Wallet,Express,WC,311.89,3.55,855.50,1082.50,Toys & Games,Outdoor Toys,Brand G,XS,Cream,3.90,60.00,3.40,0.11,15,30.00,Other,WC,2020-12-03 09:54:40,64,Mobile App,NaN,0.00,0,30,3,31,12,0,0,0.00,162.00,64,49.00,8.40,DC5,2026-04-26 13:40:05,Express,WC,25.70,1.40,0.07,0
3,ORD840003,CUST21163,2024-01-16 16:01:57,4,4,1545.93,344.97,1200.96,386.48,Card,Standard,GP,386.48,3.83,0.00,0.00,Apparel,Women's Dresses,Brand C,XXL,Navy,4.20,29.00,3.50,0.12,7,67.00,Female,GP,2021-06-06 06:11:24,31,Website,NaN,0.00,0,30,3,40,5,0,0,0.00,179.75,19,43.50,6.50,DC4,2024-01-22 00:06:50,Standard,GP,5.00,2.80,0.12,0
4,ORD840004,CUST20008,2024-06-14 07:53:53,1,1,864.55,0.00,864.55,864.55,Card,Express,FS,864.55,3.60,256.00,823.00,Home & Kitchen,Lighting,Brand Q,NaN,Blue,3.70,27.00,4.00,0.14,7,39.00,Male,FS,2022-10-18 22:09:09,19,Website,NaN,0.00,0,30,6,106,6,0,0,0.00,305.00,305,29.00,11.30,DC4,2024-06-17 06:47:23,Express,FS,40.10,4.70,0.09,0


Total modelling rows : 8,241
Unique order IDs     : 8,241

PASS: There is exactly one modelling row per order.


In [59]:
# ============================================================
# 20. ORDER-LEVEL FEATURE ENGINEERING
# ============================================================

# Order date features
model_data["order_year"] = (
    model_data["order_datetime"].dt.year
)

model_data["order_month"] = (
    model_data["order_datetime"].dt.month
)

model_data["order_day"] = (
    model_data["order_datetime"].dt.day
)

model_data["order_day_of_week"] = (
    model_data["order_datetime"].dt.dayofweek
)

model_data["order_week_of_year"] = (
    model_data["order_datetime"].dt.isocalendar().week.astype("float")
)

model_data["is_weekend"] = (
    model_data["order_day_of_week"] >= 5
).astype(int)

# Estimated delivery lead time
model_data["estimated_delivery_days"] = (
    (
        model_data["estimated_delivery_datetime"]
        - model_data["order_datetime"]
    )
    .dt.total_seconds()
    / (24 * 60 * 60)
)

# Discount-related feature
model_data["discount_fraction"] = np.where(
    model_data["order_gross_value"] > 0,
    model_data["total_promotion_amount"] /
    model_data["order_gross_value"],
    0
)

model_data["discount_fraction"] = (
    model_data["discount_fraction"]
    .clip(lower=0, upper=1)
)

print("Feature engineering completed.")

Feature engineering completed.


In [60]:
# ============================================================
# 21. REMOVE IDENTIFIERS AND POST-PURCHASE INFORMATION
# ============================================================

# These fields are identifiers rather than predictive characteristics.
identifier_columns = [
    "order_id",
    "customer_id"
]

# These fields are dates/timestamps represented by engineered features.
raw_datetime_columns = [
    "order_datetime",
    "account_creation_date",
    "estimated_delivery_datetime"
]

# Post-purchase delivery information.
# These must not be available to the model as they occur after the prediction point.
post_purchase_columns = [
    "actual_delivery_datetime",
    "delivery_attempts",
    "on_time_flag"
]

columns_to_drop = (
    identifier_columns
    + raw_datetime_columns
    + post_purchase_columns
)

# Only drop columns that exist.
columns_to_drop = [
    column
    for column in columns_to_drop
    if column in model_data.columns
]

model_data = model_data.drop(
    columns=columns_to_drop
)

print("Removed columns:")
for column in columns_to_drop:
    print(f" - {column}")

Removed columns:
 - order_id
 - customer_id
 - order_datetime
 - account_creation_date
 - estimated_delivery_datetime


In [61]:
# ============================================================
# 22. FINAL FEATURE TABLE
# ============================================================

X = model_data.drop(
    columns=["Return_Status"]
)

y = model_data["Return_Status"].astype(int)

# ------------------------------------------------------------
# Table 1: Final Feature Table Summary
# ------------------------------------------------------------

feature_summary = pd.DataFrame({
    "Metric": [
        "Number of observations",
        "Number of predictor variables"
    ],
    "Value": [
        f"{len(X):,}",
        f"{X.shape[1]:,}"
    ]
})

print("\nFINAL FEATURE TABLE SUMMARY:")
display(feature_summary)

# ------------------------------------------------------------
# Table 2: Target Distribution
# ------------------------------------------------------------

target_distribution = (
    y.value_counts()
    .sort_index()
    .rename_axis("Return_Status")
    .reset_index(name="Count")
)

target_distribution["Return_Status"] = (
    target_distribution["Return_Status"]
    .map({
        0: "0 - Not Returned",
        1: "1 - Returned"
    })
)

target_distribution["Percentage"] = (
    target_distribution["Count"]
    / target_distribution["Count"].sum()
    * 100
)

print("\nTARGET DISTRIBUTION:")

display(
    target_distribution.style
    .format({
        "Count": "{:,}",
        "Percentage": "{:.2f}%"
    })
    .hide(axis="index")
)


FINAL FEATURE TABLE SUMMARY:


,Metric,Value
0,Number of observations,"8,241"
1,Number of predictor variables,56



TARGET DISTRIBUTION:


Return_Status,Count,Percentage
0 - Not Returned,"6,372",77.32%
1 - Returned,"1,869",22.68%


In [62]:
# ============================================================
# 23. IDENTIFY NUMERIC AND CATEGORICAL FEATURES
# ============================================================

# Identify numeric and categorical features
numeric_features = X.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# ------------------------------------------------------------
# Table 1: Feature Type Summary
# ------------------------------------------------------------

feature_type_summary = pd.DataFrame({
    "Feature Type": [
        "Numeric Features",
        "Categorical Features",
        "Total Predictor Features"
    ],
    "Number of Features": [
        len(numeric_features),
        len(categorical_features),
        len(numeric_features) + len(categorical_features)
    ]
})

print("\nFEATURE TYPE SUMMARY:")
display(
    feature_type_summary.style
    .format({"Number of Features": "{:,}"})
    .hide(axis="index")
)

# ------------------------------------------------------------
# Table 2: Numeric Features
# ------------------------------------------------------------

numeric_feature_table = pd.DataFrame({
    "No.": range(1, len(numeric_features) + 1),
    "Numeric Feature": numeric_features
})

print("\nNUMERIC FEATURES:")
display(
    numeric_feature_table.style
    .hide(axis="index")
)

# ------------------------------------------------------------
# Table 3: Categorical Features
# ------------------------------------------------------------

categorical_feature_table = pd.DataFrame({
    "No.": range(1, len(categorical_features) + 1),
    "Categorical Feature": categorical_features
})

print("\nCATEGORICAL FEATURES:")
display(
    categorical_feature_table.style
    .hide(axis="index")
)


FEATURE TYPE SUMMARY:


Feature Type,Number of Features
Numeric Features,42
Categorical Features,14
Total Predictor Features,56



NUMERIC FEATURES:


No.,Numeric Feature
1,item_count
2,total_quantity
3,order_gross_value
4,total_promotion_amount
5,order_net_value
6,average_unit_price
7,average_product_price
8,average_product_rating
9,average_review_count
10,average_listing_age_days



CATEGORICAL FEATURES:


No.,Categorical Feature
1,payment_method
2,delivery_option_x
3,customer_province_x
4,product_category
5,product_subcategory
6,product_brand
7,product_size
8,product_colour
9,customer_gender
10,customer_province_y


In [63]:
# ============================================================
# 24. TRAIN / VALIDATION / UNSEEN TEST SPLIT
# ============================================================

# First split: 80% training and 20% temporary set
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# Second split: 10% validation and 10% unseen test
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# Create Dataset Split Summary Table
# ------------------------------------------------------------

split_summary = pd.DataFrame({
    "Dataset": [
        "Training Set",
        "Validation Set",
        "Unseen Test Set",
        "Total"
    ],
    "Observations": [
        len(X_train),
        len(X_validation),
        len(X_test),
        len(X_train) + len(X_validation) + len(X_test)
    ],
    "Percentage": [
        len(X_train) / len(X) * 100,
        len(X_validation) / len(X) * 100,
        len(X_test) / len(X) * 100,
        100.00
    ]
})

print("\nTRAIN / VALIDATION / UNSEEN TEST SPLIT:")

display(
    split_summary.style
    .format({
        "Observations": "{:,}",
        "Percentage": "{:.2f}%"
    })
    .hide(axis="index")
)


# ============================================================
# SAVE TRAINING HISTORY TO EXISTING EXPERIMENTAL_RESULTS
# ============================================================

from pathlib import Path

# Locate the existing project root
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in [current_path, *current_path.parents]
        if folder.name == "STADIOalot-Return-Prediction"
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the STADIOalot-Return-Prediction "
        "project root."
    )

# Use the existing folder; do not create a new one
RESULTS_PATH = PROJECT_ROOT / "experimental_results"

if not RESULTS_PATH.is_dir():
    raise FileNotFoundError(
        f"Existing experimental_results folder not found: "
        f"{RESULTS_PATH}"
    )

# Save the training history
training_history_path = (
    RESULTS_PATH / "ss3_xgboost_training_history.csv"
)

training_history_df.to_csv(
    training_history_path,
    index=True
)

print("Training history saved successfully:")
print(training_history_path.resolve())



TRAIN / VALIDATION / UNSEEN TEST SPLIT:


Dataset,Observations,Percentage
Training Set,"6,592",79.99%
Validation Set,824,10.00%
Unseen Test Set,825,10.01%
Total,"8,241",100.00%


Training history saved successfully:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\experimental_results\ss3_xgboost_training_history.csv


In [64]:
# ============================================================
# 25. VERIFY CLASS DISTRIBUTION ACROSS SPLITS
# ============================================================

split_distribution = pd.DataFrame({
    "Training": y_train.value_counts(normalize=True),
    "Validation": y_validation.value_counts(normalize=True),
    "Unseen Test": y_test.value_counts(normalize=True)
}).sort_index() * 100


split_distribution.index = [
    "0 - Not Returned",
    "1 - Returned"
]
print("Class Distribution Across Dataset Splits")
display(
    split_distribution.style
    .format("{:.2f}%")
)

Class Distribution Across Dataset Splits


,Training,Validation,Unseen Test
0 - Not Returned,77.32%,77.31%,77.33%
1 - Returned,22.68%,22.69%,22.67%


### Preprocessing and Data-Splitting Justification

The preprocessing approach is designed around the order-level prediction problem and the point at which the prediction would be made. The item-level order, product and inventory tables are aggregated to order level so that each modelling observation represents one order. This prevents a single order with multiple items from being treated as multiple independent prediction cases.

Identifiers such as `order_id` and `customer_id` are excluded as they identify records rather than represent meaningful predictive characteristics. Post-purchase delivery information is excluded as it would not be available when the return-risk prediction is generated. The return table is also excluded from the predictor set as it contains information about the outcome itself.

Numerical variables will be median-imputed and categorical variables will be most-frequent imputed and one-hot encoded. This allows the model to handle missing values and categorical characteristics without manually assigning arbitrary numerical values to categories. The data is divided into 80% training, 10% validation and 10% unseen testing data using stratified sampling. Stratification preserves approximately the same proportion of returned and non-returned orders in each subset.

The training set is used to fit the model. The validation set is used to assess model performance during development, while the unseen test set is kept separate until final evaluation. This provides a more reliable estimate of how the final model performs on previously unseen orders.

**Validation limitation:** Stratified random splitting preserves the return rate in each subset, but it does not simulate a strict future-period forecast. The extract spans multiple dates, a chronological holdout would be a useful additional robustness check before deployment: earlier orders would be used for training and later orders for validation/testing. The current split is retained for this assignment's 80/10/10 design, and the results should therefore be interpreted as performance on randomly held-out orders rather than definitive evidence of future-period performance.

# 3. Model Training

The final modelling method selected during SS2 was XGBoost (Extreme Gradient Boosting) for binary classification.

The model is used to predict:

- `0` — Not Returned
- `1` — Returned

The SS2 model configuration is retained for SS3 so that the final method is applied consistently to the new client data rather than introducing an unrelated modelling approach.

The main hyperparameters are:

| Hyperparameter | Value | Justification |
|---|---:|---|
| `n_estimators` | 300 | Provides sufficient boosting rounds to learn non-linear relationships while maintaining a practical training time. |
| `max_depth` | 6 | Allows the model to learn interactions between customer, product, seller and delivery characteristics without making individual trees excessively complex. |
| `learning_rate` | 0.05 | A relatively small learning rate makes each boosting step more conservative and supports gradual learning across the ensemble. |
| `subsample` | 0.80 | Uses 80% of observations per boosting stage, introducing randomness that can reduce overfitting. |
| `colsample_bytree` | 0.80 | Uses 80% of predictor variables for each tree, helping diversify the ensemble. |
| `min_child_weight` | 2 | Places a minimum constraint on child-node formation and helps prevent overly specific splits. |
| `reg_lambda` | 1.0 | Applies L2 regularisation to control model complexity. |
| `objective` | `binary:logistic` | Appropriate for the binary return/not-return target. |
| `eval_metric` | `logloss` | Measures the quality of the predicted probabilities during training. |
| `random_state` | 42 | Ensures reproducible results. |

The validation subset is monitored during training to observe the model's learning behaviour without using the unseen test set for model development.

The model, fitted preprocessing pipeline and training log-loss history are saved as project artifacts. The training history is exported to `experimental_results/ss3_xgboost_training_history.csv` so that training behaviour can be reviewed without relying only on notebook output.

In [65]:
# ============================================================
# 26. PREPROCESSING PIPELINE
# ============================================================

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.\n")


# ============================================================
# 27. FIT PREPROCESSING ON TRAINING DATA ONLY
# ============================================================
# Fit the preprocessor on training data only
X_train_processed = preprocessor.fit_transform(X_train)
# Transform validation and unseen test data
X_validation_processed = preprocessor.transform(
    X_validation
)
X_test_processed = preprocessor.transform(
    X_test
)

# Create Processed Dataset Shape Summary
processed_shape_summary = pd.DataFrame({
    "Dataset": [
        "Training Set",
        "Validation Set",
        "Unseen Test Set"
    ],
    "Observations": [
        X_train_processed.shape[0],
        X_validation_processed.shape[0],
        X_test_processed.shape[0]
    ],
    "Processed Features": [
        X_train_processed.shape[1],
        X_validation_processed.shape[1],
        X_test_processed.shape[1]
    ],
    "Processed Shape": [
        str(X_train_processed.shape),
        str(X_validation_processed.shape),
        str(X_test_processed.shape)
    ]
})

print("PROCESSED DATASET SHAPES:")

display(
    processed_shape_summary.style
    .format({
        "Observations": "{:,}",
        "Processed Features": "{:,}"
    })
    .hide(axis="index")
)

Preprocessing pipeline created.



PROCESSED DATASET SHAPES:


Dataset,Observations,Processed Features,Processed Shape
Training Set,"6,592",401,"(6592, 401)"
Validation Set,824,401,"(824, 401)"
Unseen Test Set,825,401,"(825, 401)"


In [66]:
# ============================================================
# 28. CREATE XGBOOST MODEL
# ============================================================

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=2,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# ------------------------------------------------------------
# Create XGBoost Model Configuration Table
# ------------------------------------------------------------

xgb_parameters = xgb_model.get_params()

xgb_config_table = pd.DataFrame({
    "Hyperparameter": [
        "n_estimators",
        "max_depth",
        "learning_rate",
        "subsample",
        "colsample_bytree",
        "min_child_weight",
        "reg_lambda",
        "objective",
        "eval_metric",
        "random_state",
        "n_jobs"
    ],
    "Value": [
        xgb_parameters["n_estimators"],
        xgb_parameters["max_depth"],
        xgb_parameters["learning_rate"],
        xgb_parameters["subsample"],
        xgb_parameters["colsample_bytree"],
        xgb_parameters["min_child_weight"],
        xgb_parameters["reg_lambda"],
        xgb_parameters["objective"],
        xgb_parameters["eval_metric"],
        xgb_parameters["random_state"],
        xgb_parameters["n_jobs"]
    ]
})

print("\nXGBOOST MODEL CONFIGURATION:")

display(
    xgb_config_table.style
    .hide(axis="index")
)

print("XGBoost model created successfully.")


XGBOOST MODEL CONFIGURATION:


Hyperparameter,Value
n_estimators,300
max_depth,6
learning_rate,0.050000
subsample,0.800000
colsample_bytree,0.800000
min_child_weight,2
reg_lambda,1.000000
objective,binary:logistic
eval_metric,logloss
random_state,42


XGBoost model created successfully.


In [67]:
# ============================================================
# 29. TRAIN XGBOOST MODEL
# ============================================================

xgb_model.fit(
    X_train_processed,
    y_train,

    eval_set=[
        (X_train_processed, y_train),
        (X_validation_processed, y_validation)
    ],

    verbose=False
)


print("XGBoost training completed.")

XGBoost training completed.


In [68]:
# ============================================================
# 30. EXTRACT TRAINING HISTORY
# ============================================================

training_history = xgb_model.evals_result()
training_history_df = pd.DataFrame({
    "Training LogLoss": training_history["validation_0"]["logloss"],
    "Validation LogLoss": training_history["validation_1"]["logloss"]
})


training_history_df.index = (
    training_history_df.index + 1
)

training_history_df.index.name = "Boosting Round"


display(
    training_history_df.head(10)
)
print(
    f"\nTotal boosting rounds: "
    f"{len(training_history_df):,}"
)

,Training LogLoss,Validation LogLoss
Boosting Round,,
1,0.53,0.53
2,0.52,0.52
3,0.51,0.52
4,0.51,0.51
5,0.50,0.51
6,0.49,0.50
7,0.49,0.50
8,0.48,0.50
9,0.48,0.49



Total boosting rounds: 300


In [69]:

# ============================================================
# 31. SAVE FINAL MODEL AND PREPROCESSOR
# ============================================================

# Define and create the models directory
MODEL_PATH = PROJECT_ROOT / "models"

MODEL_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Save the final model and fitted preprocessor
model_filenames = [
    "ss3_xgboost_return_prediction.joblib",
    "ss3_preprocessor.joblib"
]

joblib.dump(
    xgb_model,
    MODEL_PATH / model_filenames[0]
)

joblib.dump(
    preprocessor,
    MODEL_PATH / model_filenames[1]
)

# Display full directory paths in a table
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

model_file_check = pd.DataFrame({
    "File": model_filenames,
    "Full Directory": [
        str((MODEL_PATH / filename).resolve())
        for filename in model_filenames
    ],
    "Status": [
        "SAVED" if (MODEL_PATH / filename).is_file()
        else "NOT FOUND"
        for filename in model_filenames
    ]
})


print("\nFULL MODELS DIRECTORY:")
print(MODEL_PATH.resolve())

print("\nSAVED MODEL FILE CHECK:")
display(model_file_check)

# Validate saved files
missing_models = [
    filename for filename in model_filenames
    if not (MODEL_PATH / filename).is_file()
]

if missing_models:
    raise FileNotFoundError(
        "The following model files could not be found:\n"
        + "\n".join(missing_models)
    )

print("\nBoth the XGBoost model and preprocessor "
      "have been saved successfully.")


FULL MODELS DIRECTORY:
C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\models

SAVED MODEL FILE CHECK:


,File,Full Directory,Status
0,ss3_xgboost_return_prediction.joblib,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\models\ss3_xgboost_return_prediction.joblib,SAVED
1,ss3_preprocessor.joblib,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\models\ss3_preprocessor.joblib,SAVED



Both the XGBoost model and preprocessor have been saved successfully.


# 4. Model Validation

The trained XGBoost model is now applied to the validation and unseen test subsets. The unseen test subset was not used to fit the model or make modelling decisions. It therefore provides the final assessment of model performance on previously unseen orders.

The following metrics are calculated:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Accuracy measures the overall proportion of correctly classified orders.

Precision measures the proportion of orders predicted as returned that were actually returned.

Recall measures the proportion of actual returned orders that the model successfully identified.

F1-score provides a balance between precision and recall.

ROC-AUC evaluates the model's ability to distinguish between returned and non-returned orders using predicted probabilities.

In [70]:
# ============================================================
# 32. GENERATE PREDICTIONS
# ============================================================

# Training predictions
train_predictions = xgb_model.predict(X_train_processed)
train_probabilities = xgb_model.predict_proba(X_train_processed)[:, 1]

# Validation predictions
validation_predictions = xgb_model.predict(X_validation_processed)
validation_probabilities = xgb_model.predict_proba(X_validation_processed)[:, 1]

# Unseen test predictions
test_predictions = xgb_model.predict(X_test_processed)
test_probabilities = xgb_model.predict_proba(X_test_processed)[:, 1]

print(
    "Predictions generated for training, validation and unseen test data. "
    "(0 = Not Returned, 1 = Returned)"
)

# Display First 10 Predictions in 3 Separate Tables 
# Table 1: Training Predictions
train_prediction_table = pd.DataFrame({
    "Observation": range(1, min(11, len(y_train) + 1)),
    "Actual Return_Status": y_train.iloc[:10].to_numpy(dtype=int),
    "Predicted Return_Status": train_predictions[:10].astype(int),
    "Prediction Probability (%)": train_probabilities[:10] * 100
})

print("\nTABLE 1: FIRST 10 TRAINING PREDICTIONS")

display(
    train_prediction_table.style
    .format({"Prediction Probability (%)": "{:.2f}%"})
    .hide(axis="index")
)

# Table 2: Validation Predictions
validation_prediction_table = pd.DataFrame({
    "Observation": range(1, min(11, len(y_validation) + 1)),
    "Actual Return_Status": y_validation.iloc[:10].to_numpy(dtype=int),
    "Predicted Return_Status": validation_predictions[:10].astype(int),
    "Prediction Probability (%)": validation_probabilities[:10] * 100
})

print("\nTABLE 2: FIRST 10 VALIDATION PREDICTIONS")

display(
    validation_prediction_table.style
    .format({"Prediction Probability (%)": "{:.2f}%"})
    .hide(axis="index")
)

# Table 3: Unseen Test Predictions
test_prediction_table = pd.DataFrame({
    "Observation": range(1, min(11, len(y_test) + 1)),
    "Actual Return_Status": y_test.iloc[:10].to_numpy(dtype=int),
    "Predicted Return_Status": test_predictions[:10].astype(int),
    "Prediction Probability (%)": test_probabilities[:10] * 100
})

print("\nTABLE 3: FIRST 10 UNSEEN TEST PREDICTIONS")

display(
    test_prediction_table.style
    .format({"Prediction Probability (%)": "{:.2f}%"})
    .hide(axis="index")
)

Predictions generated for training, validation and unseen test data. (0 = Not Returned, 1 = Returned)

TABLE 1: FIRST 10 TRAINING PREDICTIONS


Observation,Actual Return_Status,Predicted Return_Status,Prediction Probability (%)
1,0,0,17.02%
2,0,0,7.41%
3,0,0,3.69%
4,1,1,58.49%
5,1,1,82.42%
6,1,1,84.37%
7,1,1,72.47%
8,1,1,91.12%
9,0,0,6.06%
10,1,0,43.76%



TABLE 2: FIRST 10 VALIDATION PREDICTIONS


Observation,Actual Return_Status,Predicted Return_Status,Prediction Probability (%)
1,0,0,11.60%
2,0,0,2.92%
3,0,0,13.50%
4,1,0,36.99%
5,0,0,19.01%
6,0,0,14.45%
7,0,0,25.41%
8,0,0,2.60%
9,0,0,5.91%
10,1,1,59.93%



TABLE 3: FIRST 10 UNSEEN TEST PREDICTIONS


Observation,Actual Return_Status,Predicted Return_Status,Prediction Probability (%)
1,0,0,6.03%
2,1,0,44.82%
3,0,0,3.40%
4,0,1,69.39%
5,1,0,40.17%
6,0,1,56.31%
7,0,0,3.82%
8,0,0,7.17%
9,0,0,4.51%
10,0,0,7.83%


### Decision Threshold Analysis

The model's default classification threshold is 0.50. Since returned orders are the minority class, alternative thresholds are compared using the **validation subset only**. This illustrates the trade-off between precision and recall. The unseen test set is not used to select a threshold, and the primary reported test metrics below remain based on the model's default threshold for a transparent baseline.

In [71]:
# ============================================================
# 33. VALIDATION-ONLY DECISION THRESHOLD ANALYSIS
# ============================================================

# The default classification threshold is 0.50. As returned orders
# are the minority class, evaluate alternative thresholds on the
# validation set only. Do not select a threshold using the unseen test set.

threshold_values = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

threshold_results = []
for threshold in threshold_values:
    validation_threshold_predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_validation,
            validation_threshold_predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_validation,
            validation_threshold_predictions,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_validation,
            validation_threshold_predictions,
            zero_division=0
        ),
        "Predicted Returned Orders": int(
            validation_threshold_predictions.sum()
        )
    })

threshold_analysis_table = pd.DataFrame(threshold_results)

print("Decision Threshold Comparison — Validation Set Only")
display(
    threshold_analysis_table.style
    .format({
        "Threshold": "{:.2f}",
        "Precision": "{:.2f}",
        "Recall": "{:.2f}",
        "F1 Score": "{:.2f}",
        "Predicted Returned Orders": "{:,}"
    })
    .hide(axis="index")
)

print(
    "Interpretation: a lower threshold may identify more returned orders "
    "but can also increase false positives. Select an operational threshold "
    "using validation results and the business costs of missed returns "
    "versus unnecessary interventions. The unseen test set is not used "
    "to choose the threshold."
)


Decision Threshold Comparison — Validation Set Only


Threshold,Precision,Recall,F1 Score,Predicted Returned Orders
0.20,0.38,0.61,0.47,297
0.25,0.43,0.55,0.48,239
0.30,0.48,0.49,0.48,193
0.35,0.51,0.42,0.46,153
0.40,0.57,0.36,0.44,117
0.45,0.60,0.29,0.39,92
0.50,0.62,0.23,0.34,69


Interpretation: a lower threshold may identify more returned orders but can also increase false positives. Select an operational threshold using validation results and the business costs of missed returns versus unnecessary interventions. The unseen test set is not used to choose the threshold.


In [ ]:
# ============================================================
# 34. MODEL EVALUATION FUNCTION
# ============================================================

def calculate_metrics(actual, predicted, probabilities):
    return {
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(
            actual, predicted, zero_division=0
        ),
        "Recall": recall_score(
            actual, predicted, zero_division=0
        ),
        "F1 Score": f1_score(
            actual, predicted, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(actual, probabilities)
    }

# Display Model Evaluation Metrics in a Table 
evaluation_table = pd.DataFrame({
    "Training": calculate_metrics(
        y_train,
        train_predictions,
        train_probabilities
    ),
    "Validation": calculate_metrics(
        y_validation,
        validation_predictions,
        validation_probabilities
    ),
    "Unseen Test": calculate_metrics(
        y_test,
        test_predictions,
        test_probabilities
    )
}).T

evaluation_table.index.name = "Dataset"
evaluation_table = evaluation_table.reset_index()

# Display results to two decimal places
print("XGBoost Model Evaluation Metrics")
display(
    evaluation_table.style
    .format({
        "Accuracy": "{:.2f}",
        "Precision": "{:.2f}",
        "Recall": "{:.2f}",
        "F1 Score": "{:.2f}",
        "ROC-AUC": "{:.2f}"
    })
    .hide(axis="index")
)


# ============================================================
# 38. SAVE METRICS TO EXISTING EXPERIMENTAL_RESULTS FOLDER
# ============================================================

# Locate the existing project root
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in [current_path, *current_path.parents]
        if folder.name == "STADIOalot-Return-Prediction"
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the STADIOalot-Return-Prediction "
        "project root."
    )

# Use the existing experimental_results folder
RESULTS_PATH = PROJECT_ROOT / "experimental_results"

if not RESULTS_PATH.is_dir():
    raise FileNotFoundError(
        f"Existing experimental_results folder not found: "
        f"{RESULTS_PATH}"
    )

# Save evaluation metrics
metrics_path = RESULTS_PATH / "ss3_xgboost_metrics.csv"
metrics_table.to_csv(metrics_path, index=True)

# Save confusion matrix
confusion_matrix_path = (
    RESULTS_PATH / "ss3_xgboost_confusion_matrix.csv"
)
confusion_matrix_table.to_csv(
    confusion_matrix_path,
    index=True
)

# Check that all three files exist
saved_files = [
    metrics_path,
    confusion_matrix_path,
    RESULTS_PATH / "ss3_xgboost_training_history.csv"
]

saved_files_table = pd.DataFrame({
    "File": [file.name for file in saved_files],
    "Full Directory": [
        str(file.resolve()) for file in saved_files
    ],
    "Status": [
        "SAVED" if file.is_file() else "NOT FOUND"
        for file in saved_files
    ]
})

print("EXISTING EXPERIMENTAL_RESULTS DIRECTORY:")
print(RESULTS_PATH.resolve())

display(saved_files_table.style.hide(axis="index"))

# Raise an error if any expected file is missing
missing_files = [
    file.name for file in saved_files
    if not file.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "The following files were not found:\n"
        + "\n".join(missing_files)
    )

print("\nAll three result files are saved successfully.")


XGBoost Model Evaluation Metrics


Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
Training,0.96,1.00,0.81,0.90,1.00
Validation,0.79,0.62,0.23,0.34,0.74
Unseen Test,0.78,0.56,0.20,0.29,0.70


NameError: name 'metrics_table' is not defined

In [ ]:
# ============================================================
# 35. CALCULATE TRAINING, VALIDATION AND UNSEEN TEST METRICS
# ============================================================

train_metrics = calculate_metrics(
    y_train,
    train_predictions,
    train_probabilities
)

validation_metrics = calculate_metrics(
    y_validation,
    validation_predictions,
    validation_probabilities
)

test_metrics = calculate_metrics(
    y_test,
    test_predictions,
    test_probabilities
)

metrics_table = pd.DataFrame({
    "Training": train_metrics,
    "Validation": validation_metrics,
    "Unseen Test": test_metrics
})

print("XGBoost Performance Across Dataset Splits")
display(
    metrics_table.style
    .format("{:.2f}")
)

In [ ]:
# ============================================================
# 36. UNSEEN TEST CLASSIFICATION REPORT
# ============================================================

print("Classification Report — Unseen Test Set")

# Generate classification report
report = classification_report(
    y_test,
    test_predictions,
    target_names=[
        "Not Returned",
        "Returned"
    ],
    output_dict=True,
    zero_division=0
)

# Convert report to a DataFrame
classification_report_table = pd.DataFrame(report).T

# Rename columns
classification_report_table = classification_report_table.rename(
    columns={
        "precision": "Precision (%)",
        "recall": "Recall (%)",
        "f1-score": "F1 Score (%)",
        "support": "Support"
    }
)

# Convert metrics to percentages
for column in ["Precision (%)", "Recall (%)", "F1 Score (%)"]:
    classification_report_table[column] = (
        classification_report_table[column] * 100
    )

# Calculate support percentage
classification_report_table["Support (%)"] = (
    classification_report_table["Support"] / len(y_test) * 100
)

# Remove raw support counts
classification_report_table = classification_report_table.drop(
    columns=["Support"]
)

# Display percentage-only table
display(
    classification_report_table.style
    .format("{:.2f}%", na_rep="—")
)

In [ ]:
# ============================================================
# 37. CONFUSION MATRIX — UNSEEN TEST SET
# ============================================================

cm = confusion_matrix(
    y_test,
    test_predictions
)


confusion_matrix_table = pd.DataFrame(
    cm,
    index=[
        "Actual: Not Returned",
        "Actual: Returned"
    ],
    columns=[
        "Predicted: Not Returned",
        "Predicted: Returned"
    ]
)

print("Confusion Matrix — Unseen Test Set")
display(
    confusion_matrix_table.style
    .format("{:,}")
)

In [ ]:
# ============================================================
# 38. SAVE METRICS TO EXPERIMENTAL_RESULTS FOLDER
# ============================================================

from pathlib import Path

# Set the project root
PROJECT_ROOT = Path.cwd().resolve()

# Locate the project root if running from a subfolder
for folder in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if folder.name == "STADIOalot-Return-Prediction":
        PROJECT_ROOT = folder
        break

# Define the results directory
RESULTS_PATH = PROJECT_ROOT / "experimental_results"

# Create the folder if it does not exist
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

# Save evaluation metrics
metrics_table.to_csv(
    RESULTS_PATH / "ss3_xgboost_metrics.csv",
    index=True
)

# Save confusion matrix
confusion_matrix_table.to_csv(
    RESULTS_PATH / "ss3_xgboost_confusion_matrix.csv",
    index=True
)

# Display saved file locations in a table
saved_files_table = pd.DataFrame({
    "File": [
        "ss3_xgboost_metrics.csv",
        "ss3_xgboost_confusion_matrix.csv"
    ],
    "Full Directory": [
        str((RESULTS_PATH / "ss3_xgboost_metrics.csv").resolve()),
        str((RESULTS_PATH / "ss3_xgboost_confusion_matrix.csv").resolve())
    ],
    "Status": [
        "Saved successfully",
        "Saved successfully"
    ]
})

display(saved_files_table.style.hide(axis="index"))

# 5. Results Visualisation


The visualisations provide a graphical summary of both the model's training behaviour and its performance on unseen data.
The following visualisations are included:

1. Training versus validation log-loss across boosting rounds.
2. Accuracy, precision, recall and F1-score across training, validation and unseen test subsets.
3. Confusion matrix for the unseen test subset.
4. ROC curve and ROC-AUC for the unseen test subset.

The figures are given sufficient spacing and readable labels so that the results can be interpreted clearly.

In [ ]:
# ============================================================
# 39. VISUALISATION 1
# TRAINING VS VALIDATION LOG-LOSS
# ============================================================

plt.figure(figsize=(12, 6))

plt.plot(
    training_history_df.index,
    training_history_df["Training LogLoss"],
    label="Training LogLoss",
    linewidth=2,
    color="#7e03a8"
)

plt.plot(
    training_history_df.index,
    training_history_df["Validation LogLoss"],
    label="Validation LogLoss",
    linewidth=2,
    color="#fca636"
)

plt.xlabel(
    "Boosting Round",
    fontsize=12
)

plt.ylabel(
    "Log Loss",
    fontsize=12
)

plt.title(
    "XGBoost Training and Validation Log Loss",
    fontsize=15,
    pad=15
)

plt.legend(
    fontsize=10
)

plt.grid(
    alpha=0.3
)

plt.tight_layout(
    pad=2
)

plt.show()

In [ ]:
# ============================================================
# 40. VISUALISATION 2
# PERFORMANCE ACROSS DATASET SPLITS
# ============================================================

# ------------------------------------------------------------
# PREPARE METRICS FOR VISUALISATION
# ------------------------------------------------------------

required_metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

required_splits = [
    "Training",
    "Validation",
    "Unseen Test"
]

# Check whether dataset splits are stored as columns or rows
if all(split in metrics_table.columns for split in required_splits):

    plot_metrics = metrics_table.loc[
        required_metrics,
        required_splits
    ].T

elif all(split in metrics_table.index for split in required_splits):

    plot_metrics = metrics_table.loc[
        required_splits,
        required_metrics
    ]

else:
    print("Current metrics_table structure:")
    display(metrics_table)

    raise ValueError(
        "The dataset splits or metric names do not match the expected "
        "structure. Check the displayed metrics_table."
    )

# ------------------------------------------------------------
# CREATE BAR CHART
# ------------------------------------------------------------

ax = plot_metrics.plot(
    kind="bar",
    figsize=(13, 7),
    width=0.75,
    colormap="plasma"
)

ax.set_title(
    "XGBoost Performance Across Dataset Splits",
    fontsize=15,
    pad=15
)

ax.set_xlabel(
    "Dataset Split",
    fontsize=12
)

ax.set_ylabel(
    "Score",
    fontsize=12
)

# Leave sufficient space above the bars for value labels
ax.set_ylim(0, 1.15)

# Add values above each bar
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=4,
        fontsize=10
    )

ax.legend(
    title="Evaluation Metric",
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

ax.grid(
    axis="y",
    alpha=0.3
)

# Keep gridlines behind the bars and labels
ax.set_axisbelow(True)

plt.xticks(rotation=0)
plt.tight_layout(pad=2)
plt.show()

In [ ]:
# ============================================================
# 41. VISUALISATION 3
# CONFUSION MATRIX
# ============================================================

plt.figure(
    figsize=(8, 7)
)

plt.imshow(
    cm,
    interpolation="nearest",
    cmap="viridis"
)

plt.title(
    "Confusion Matrix — Unseen Test Set",
    fontsize=15,
    pad=15
)

plt.colorbar(
    fraction=0.046,
    pad=0.04
)

class_names = [
    "Not Returned",
    "Returned"
]

plt.xticks(
    [0, 1],
    class_names,
    rotation=0
)

plt.yticks(
    [0, 1],
    class_names
)

plt.xlabel(
    "Predicted Class",
    fontsize=12
)

plt.ylabel(
    "Actual Class",
    fontsize=12
)

# Add values inside the matrix
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            f"{cm[i, j]:,}",
            ha="center",
            va="center",
            fontsize=14,
            color="white"
        )

plt.tight_layout(
    pad=2
)

plt.show()

In [ ]:
# ============================================================
# 42. VISUALISATION 4
# ROC CURVE — UNSEEN TEST SET
# ============================================================

fpr, tpr, thresholds = roc_curve(
    y_test,
    test_probabilities
)

test_auc = roc_auc_score(
    y_test,
    test_probabilities
)

plt.figure(figsize=(10, 7))

# ROC curve
plt.plot(
    fpr,
    tpr,
    linewidth=2.5,
    color="#7e03a8",
    label=f"XGBoost ROC-AUC = {test_auc:.2f}"
)

# Random classifier baseline
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    color="#fca636",
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate",
    fontsize=12
)

plt.ylabel(
    "True Positive Rate",
    fontsize=12
)

plt.title(
    "ROC Curve — Unseen Test Set",
    fontsize=15,
    pad=15
)

plt.legend(
    loc="lower right"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout(pad=2)
plt.show()

## Interpretation of Results

The training, validation and unseen-test results should be interpreted together rather than relying on accuracy alone.

The training-versus-validation log-loss plot is used to assess the model's learning behaviour. A substantial gap between training and validation performance indicates possible overfitting, meaning the model fits the training data much better than data it has not seen. In the current results, training ROC-AUC is approximately 1.00, compared with 0.74 on validation and 0.70 on the unseen test set. This gap is consistent with overfitting and suggests that generalisation needs improvement.

The metric comparison chart shows how accuracy, precision, recall and F1-score differ across the training, validation and unseen-test subsets. The confusion matrix identifies the numbers of true negatives, false positives, false negatives and true positives on the unseen test set. The ROC curve summarises discrimination across probability thresholds; ROC-AUC should be considered alongside class-specific precision and recall.

### Business Interpretation of Unseen-Test Results

The XGBoost model achieved approximately **78.30% accuracy** and a **ROC-AUC of 0.70** on the unseen test set. Accuracy must be interpreted in relation to the class distribution: approximately **77.33%** of test orders were not returned, so a majority-class baseline would achieve about 77.33% accuracy. The model therefore provides only a modest improvement in overall accuracy over this simple baseline.

The model's recall for returned orders was approximately **19.79%**, meaning it identified only a small proportion of orders that actually resulted in returns. The confusion matrix shows that **150 returned orders were incorrectly classified as not returned**, while only **37 returned orders were correctly identified**. This is an important limitation as the business objective is to identify potentially return-prone orders.

The results suggest that the model has some ability to distinguish between returned and non-returned orders, but its current classification performance may not be sufficient for operational use. The validation-only threshold analysis above demonstrates how alternative thresholds affect precision and recall. Any operational threshold should be selected according to the business costs of missed returns and unnecessary interventions, using validation data rather than the unseen test set. Further work should also investigate class-imbalance strategies, precision-recall analysis and chronological validation to assess performance on later orders.

In [ ]:
# ============================================================
# 43. UNSEEN TEST RESULTS
# ============================================================

final_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ],
    "Unseen Test Score": [
        test_metrics["Accuracy"],
        test_metrics["Precision"],
        test_metrics["Recall"],
        test_metrics["F1 Score"],
        test_metrics["ROC-AUC"]
    ]
})

print("XGBoost Performance on Unseen Test Data")
display(
    final_results.style
    .format({
        "Unseen Test Score": "{:.2f}"
    })
)

## Conclusion

The implementation applies the final XGBoost return-prediction method selected during SS2 to the supplied STADIOalot client extract.

The eight client tables were combined into an order-level modelling dataset. Item-level information was aggregated so that each order represents one prediction instance. The preprocessing process addresses data types, missing values and categorical variables while excluding identifiers and information that would only become available after the prediction point.

The model was trained using the SS2-selected XGBoost configuration and evaluated on separate training, validation and unseen-test subsets. The model, fitted preprocessor, evaluation metrics, confusion matrix and training history are saved to the project folders for reproducibility.

The unseen-test results show approximately 78.30% accuracy and 0.70 ROC-AUC, but recall for returned orders is only about 19.79%. The training-to-test performance gap also suggests overfitting. Consequently, the current model should be treated as a prototype rather than a deployment-ready solution. Further evaluation should focus on improving identification of returned orders, assessing validation-based decision thresholds, and testing performance with a chronological holdout before operational use.